In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- GPU guard: stop before spending quota if Kaggle gave us the wrong accelerator ---
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", names)
if not names or not all("T4" in n for n in names):
    raise SystemExit(f"Expected T4 GPUs, got {names}. Set Accelerator -> 'GPU T4 x2' and run again.")


In [ ]:
# --- locate the tokenizer and all token-shard dirs among the attached kernel outputs ---
DATA_DIRS = inputs_with("manifest.json", "vishalam-data-")
TOKENIZER = inputs_with("tokenizer.json", "vishalam-data-1")[0] + "/tokenizer.json"
print("shard dirs:", DATA_DIRS)


In [ ]:
# --- Pretraining session: resume from the other ping-pong kernel's output, stop cleanly before 12 h ---
prev = inputs_with("run/ckpt.pt", "vishalam-pretrain-350m-")
resume = f"--resume-from {prev[0]}/run" if prev else ""
print("resuming from:", prev or "scratch")
n = torch.cuda.device_count()
launcher = f"torchrun --standalone --nproc_per_node={n}" if n > 1 else "python"
get_ipython().system(f"{launcher} -m vishalam.train.pretrain --config configs/vishalam-350m.yaml "
                     f"--data-dirs {','.join(DATA_DIRS)} --tokenizer {TOKENIZER} {resume} "
                     f"--out {WORK}/run --max-hours {hours_left():.2f}")


In [ ]:
# --- Loss curve so far ---
import pandas as pd
df = pd.read_csv(f"{WORK}/run/log.csv")
ax = df.plot(x="tokens", y="loss", logy=True, figsize=(9, 4))
v = df.dropna(subset=["val_loss"])
if len(v): v.plot(x="tokens", y="val_loss", ax=ax, style="o-")
df.tail()
